# K-mer Bleed Deconvolution Benchmark

Compares three approaches for suppressing **k-mer signal bleed** in nanopore modification detection.

## Problem

A single modification at position *p* perturbs the signals for all 5-mers that overlap it (positions *p−2* to *p+2*).
This causes elevated `mod_ratio` at flanking positions that are **not** truly modified.

## Methods Compared

| Label | Approach | Idea |
|-------|----------|------|
| `original` | No correction | Baseline — raw pipeline output |
| `peak_calling` | Local peak selection | Keep only the highest position in each 5-mer window |
| `lasso` | L1-regularized deconvolution | Model `mod_ratio = X @ β` where X is the k-mer overlap matrix, solve with LASSO |
| `read_consistency` | Per-read k-mer consistency | Attenuate positions that aren't the local max on each read |

## 1. Imports and Configuration

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.linear_model import Lasso
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

from baleen.eventalign import load_results
from baleen.eventalign._hierarchical import compute_sequential_modification_probabilities
from baleen.eventalign._aggregation import aggregate_contig, aggregate_all

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

KMER_K = 5  # nanopore k-mer length
HALF_K = KMER_K // 2  # = 2 for 5-mer

## 2. File Paths — Edit These

In [ ]:
BALEEN_PKL = Path('../output/pipeline_results.pkl')
POSITION_OFFSET = 0  # set to 0 if positions are already 1-based center-of-kmer

## 3. Ground Truth: Known Modification Sites (E. coli rRNA)

In [ ]:
KNOWN_MODIFICATIONS = {
    # Pseudouridine (Ψ)
    ('ecoli16S', 516):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 746):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 955):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 1911): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 1917): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2457): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2504): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2580): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2604): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2605): ('Ψ',     'pseudouridine'),
    # N2-methylguanosine (m2G)
    ('ecoli16S', 966):  ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1207): ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1516): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 1835): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 2445): ('m2G',   'N2-methylguanosine'),
    # 5-methylcytidine (m5C)
    ('ecoli16S', 967):  ('m5C',   '5-methylcytidine'),
    ('ecoli16S', 1407): ('m5C',   '5-methylcytidine'),
    ('ecoli23S', 1962): ('m5C',   '5-methylcytidine'),
    # 5-methyluridine (m5U / ribothymidine)
    ('ecoli23S', 747):  ('m5U',   '5-methyluridine'),
    ('ecoli23S', 1939): ('m5U',   '5-methyluridine'),
    # N6,N6-dimethyladenosine (m6,6A)
    ('ecoli16S', 1518): ('m6,6A', 'N6,N6-dimethyladenosine'),
    ('ecoli16S', 1519): ('m6,6A', 'N6,N6-dimethyladenosine'),
    # N6-methyladenosine (m6A)
    ('ecoli23S', 1618): ('m6A',   'N6-methyladenosine'),
    ('ecoli23S', 2030): ('m6A',   'N6-methyladenosine'),
    # N7-methylguanosine (m7G)
    ('ecoli16S', 527):  ('m7G',   'N7-methylguanosine'),
    ('ecoli23S', 2069): ('m7G',   'N7-methylguanosine'),
    # Other modifications
    ('ecoli23S', 2498): ('Cm',    "2'-O-methylcytosine"),
    ('ecoli23S', 2449): ('D',     'dihydrouridine'),
    ('ecoli23S', 2251): ('Gm',    "2'-O-methylguanosine"),
    ('ecoli23S', 2552): ('Um',    "2'-O-methyluridine"),
    ('ecoli23S', 2501): ('ho5C',  '5-hydroxycytidine'),
    ('ecoli23S', 745):  ('m1G',   '1-methylguanosine'),
    ('ecoli23S', 2503): ('m2A',   '2-methyladenosine'),
    ('ecoli23S', 1915): ('m3Ψ',   '3-methylpseudouridine'),
    ('ecoli16S', 1498): ('m3U',   '3-methyluridine'),
    ('ecoli16S', 1402): ('m4Cm',  "N4,2'-O-dimethylcytidine"),
}

KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_SET = set(KNOWN_MOD_PIPELINE.keys())

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f'Total known modification sites: {len(KNOWN_MODIFICATIONS)}')
print(f'Modification types: {len(mod_counts)}')
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f'  {mod_type:8s}  {count} sites   ({full})')

## 4. Load Pipeline Results and Run Hierarchical Stages

In [ ]:
contig_results = None
for candidate in [BALEEN_PKL,
                  Path('../results/pipeline_results.pkl'),
                  Path('../output/pipeline_results.pkl'),
                  Path('output/pipeline_results.pkl')]:
    if candidate.exists():
        print(f'Loading: {candidate}')
        contig_results = load_results(candidate)
        break

if contig_results is None:
    raise FileNotFoundError(
        'No pipeline_results.pkl found. Run `baleen run ...` first, '
        'then set BALEEN_PKL to the output path.'
    )

for contig, cr in contig_results.items():
    print(f'{contig}: {len(cr.positions)} positions')

In [ ]:
contig_results = None
metadata = None
for candidate in [BALEEN_PKL,
                  Path('../results/pipeline_results.pkl'),
                  Path('../output/pipeline_results.pkl'),
                  Path('output/pipeline_results.pkl')]:
    if candidate.exists():
        print(f'Loading: {candidate}')
        contig_results, metadata = load_results(candidate)
        break

if contig_results is None:
    raise FileNotFoundError(
        'No pipeline_results.pkl found. Run `baleen run ...` first, '
        'then set BALEEN_PKL to the output path.'
    )

for contig, cr in contig_results.items():
    print(f'{contig}: {len(cr.positions)} positions')

## 5. Build Original Site-Level DataFrame

In [ ]:
# Aggregate using the standard pipeline
site_results_orig = aggregate_all(hier_results, score_field='p_mod_hmm')

df = pd.DataFrame([
    {
        'contig': sr.contig,
        'position': sr.position,
        'kmer': sr.kmer,
        'mod_ratio_orig': sr.mod_ratio,
        'pvalue_orig': sr.pvalue,
        'padj_orig': sr.padj,
        'effect_size_orig': sr.effect_size,
        'n_native': sr.n_native,
        'n_ivt': sr.n_ivt,
        'mean_p_mod_orig': sr.mean_p_mod,
    }
    for sr in site_results_orig
])

# Add ground-truth labels
df['y_true'] = df.apply(
    lambda r: 1 if (r['contig'], r['position']) in KNOWN_MOD_SET else 0,
    axis=1,
)
df['mod_type'] = df.apply(
    lambda r: KNOWN_MOD_PIPELINE.get((r['contig'], r['position']), ('unmod', ''))[0],
    axis=1,
)

print(f'Total sites: {len(df)}')
print(f'Known modified: {df["y_true"].sum()}')
print(f'Unmodified: {(df["y_true"] == 0).sum()}')
df.head(10)

---
## 6. Method A: Peak-Calling Deconvolution

Simple approach: within each sliding window of `KMER_K` consecutive positions,
if multiple positions have elevated `mod_ratio`, keep only the peak and suppress the flanks.

Two variants:
- **Hard**: flanking positions zeroed out
- **Soft**: flanking positions attenuated by distance from peak

In [ ]:
def peak_calling_deconv(positions, mod_ratios, kmer_k=5, threshold=0.1, mode='soft'):
    """Suppress k-mer bleed by local peak selection.

    Parameters
    ----------
    positions : array of int, sorted
    mod_ratios : array of float, same length as positions
    kmer_k : int, k-mer length
    threshold : float, minimum mod_ratio to consider as a potential peak
    mode : 'hard' or 'soft'
        'hard': flanking positions set to 0
        'soft': flanking positions attenuated by (1 - distance/half_k)^2

    Returns
    -------
    adjusted : array of float, same shape as mod_ratios
    """
    n = len(positions)
    adjusted = mod_ratios.copy()
    half_k = kmer_k // 2

    # Build position → index lookup
    pos_to_idx = {p: i for i, p in enumerate(positions)}

    # Find peaks: positions that are local maxima in their k-mer neighborhood
    is_peak = np.zeros(n, dtype=bool)
    for i in range(n):
        if mod_ratios[i] < threshold:
            continue
        # Check if this is the max within ±half_k bases
        is_max = True
        for offset in range(-half_k, half_k + 1):
            if offset == 0:
                continue
            neighbor_pos = positions[i] + offset
            j = pos_to_idx.get(neighbor_pos)
            if j is not None and mod_ratios[j] > mod_ratios[i]:
                is_max = False
                break
        is_peak[i] = is_max

    # Suppress non-peak positions that are within half_k of a peak
    for i in range(n):
        if is_peak[i] or mod_ratios[i] < threshold:
            continue
        # Check if any neighbor within ±half_k is a peak with higher score
        nearest_peak_dist = None
        for offset in range(-half_k, half_k + 1):
            if offset == 0:
                continue
            neighbor_pos = positions[i] + offset
            j = pos_to_idx.get(neighbor_pos)
            if j is not None and is_peak[j] and mod_ratios[j] >= mod_ratios[i]:
                dist = abs(offset)
                if nearest_peak_dist is None or dist < nearest_peak_dist:
                    nearest_peak_dist = dist

        if nearest_peak_dist is not None:
            if mode == 'hard':
                adjusted[i] = 0.0
            else:  # soft
                # Quadratic attenuation: closer to peak → more suppression
                suppression = (1.0 - nearest_peak_dist / (half_k + 1)) ** 2
                adjusted[i] *= (1.0 - suppression)

    return adjusted


# Apply per contig
df['mod_ratio_peak_hard'] = 0.0
df['mod_ratio_peak_soft'] = 0.0

for contig in df['contig'].unique():
    mask = df['contig'] == contig
    sub = df[mask].sort_values('position')
    positions = sub['position'].values
    ratios = sub['mod_ratio_orig'].values

    hard = peak_calling_deconv(positions, ratios, kmer_k=KMER_K, mode='hard')
    soft = peak_calling_deconv(positions, ratios, kmer_k=KMER_K, mode='soft')

    df.loc[sub.index, 'mod_ratio_peak_hard'] = hard
    df.loc[sub.index, 'mod_ratio_peak_soft'] = soft

print('Peak-calling deconvolution applied.')
print(f'  Hard: {(df["mod_ratio_peak_hard"] == 0).sum()} positions zeroed')
print(f'  Soft: mean attenuation = {1 - df["mod_ratio_peak_soft"].sum() / df["mod_ratio_orig"].sum():.3f}')

---
## 7. Method B: LASSO K-mer Deconvolution

Model: each k-mer position's `mod_ratio` is the sum of per-base modification effects
for the bases it covers. Solve with L1 regularization to get sparse per-base effects.

$$y_{\text{pos}} = \sum_{j \in \text{bases covered}} \beta_j + \varepsilon$$

$$\min_\beta \|y - X\beta\|^2 + \lambda \|\beta\|_1$$

In [ ]:
def lasso_deconv(positions, mod_ratios, kmer_k=5, alpha=0.01):
    """LASSO deconvolution of k-mer overlap signal bleed.

    Parameters
    ----------
    positions : array of int, sorted genomic positions (k-mer center)
    mod_ratios : array of float, observed mod_ratio per position
    kmer_k : int, k-mer length
    alpha : float, LASSO regularization strength (larger = more sparse)

    Returns
    -------
    adjusted : array of float, deconvolved mod_ratio per position
    beta : dict mapping base_position → estimated per-base effect
    """
    half_k = kmer_k // 2
    n_pos = len(positions)

    if n_pos < 3:
        return mod_ratios.copy(), {}

    # Determine all base positions covered by any k-mer
    all_bases = set()
    for p in positions:
        for offset in range(-half_k, half_k + 1):
            all_bases.add(p + offset)
    all_bases = sorted(all_bases)
    base_to_col = {b: j for j, b in enumerate(all_bases)}
    n_bases = len(all_bases)

    # Build coverage matrix X: shape (n_positions, n_bases)
    X = np.zeros((n_pos, n_bases), dtype=np.float64)
    for i, p in enumerate(positions):
        for offset in range(-half_k, half_k + 1):
            base_pos = p + offset
            if base_pos in base_to_col:
                X[i, base_to_col[base_pos]] = 1.0

    # Normalize X columns so that regularization is balanced
    # (each base appears in at most kmer_k positions)
    y = mod_ratios.copy()

    # Fit LASSO
    lasso = Lasso(alpha=alpha, positive=True, max_iter=10000, tol=1e-6)
    lasso.fit(X, y)
    beta_vec = lasso.coef_

    # Map back: for each position, its deconvolved score is just
    # the beta at its CENTER base (the position itself)
    adjusted = np.zeros(n_pos, dtype=np.float64)
    beta_dict = {}
    for j, base_pos in enumerate(all_bases):
        beta_dict[base_pos] = beta_vec[j]

    for i, p in enumerate(positions):
        adjusted[i] = beta_dict.get(p, 0.0)

    return adjusted, beta_dict


# Try multiple alpha values
ALPHA_VALUES = [0.001, 0.005, 0.01, 0.02, 0.05]

for alpha in ALPHA_VALUES:
    col = f'mod_ratio_lasso_{alpha}'
    df[col] = 0.0

    for contig in df['contig'].unique():
        mask = df['contig'] == contig
        sub = df[mask].sort_values('position')
        positions = sub['position'].values
        ratios = sub['mod_ratio_orig'].values

        adj, _ = lasso_deconv(positions, ratios, kmer_k=KMER_K, alpha=alpha)
        df.loc[sub.index, col] = adj

    n_nonzero = (df[col] > 0.01).sum()
    print(f'LASSO α={alpha}: {n_nonzero} non-zero positions (out of {len(df)})')

---
## 8. Method C: Read-Level K-mer Consistency

For each read at each position, check if this position is the local maximum
within the k-mer neighborhood along that read's trajectory.
If not, attenuate the read's contribution.

In [ ]:
def read_consistency_deconv(hier_results, contig_results, kmer_k=5):
    """Per-read k-mer consistency adjustment.

    For each read at each position, compare its p_mod_hmm to its p_mod_hmm
    at neighboring positions. If the current position is NOT the read-level
    local max, attenuate its p_mod_hmm.

    Returns
    -------
    adjusted_scores : dict of {(contig, position): adjusted_mean_p_mod}
    """
    half_k = kmer_k // 2
    results = {}

    for contig, cmr in hier_results.items():
        sorted_positions = sorted(cmr.position_stats.keys())
        pos_set = set(sorted_positions)

        # Build read → {position: (read_index, p_mod_hmm)} from trajectories
        read_pos_scores = {}  # read_name → {pos: p_mod_hmm}

        for traj in cmr.native_trajectories:
            read_pos_scores[traj.read_name] = {}
            for pos, idx in zip(traj.positions, traj.indices):
                ps = cmr.position_stats.get(pos)
                if ps is not None and not np.isnan(ps.p_mod_hmm[idx]):
                    read_pos_scores[traj.read_name][pos] = float(ps.p_mod_hmm[idx])

        # For each position, compute adjusted native scores
        for pos in sorted_positions:
            ps = cmr.position_stats[pos]
            n_native = ps.n_native
            if n_native == 0:
                results[(contig, pos)] = 0.0
                continue

            adjusted_native = []

            for traj in cmr.native_trajectories:
                if pos not in read_pos_scores.get(traj.read_name, {}):
                    continue
                my_score = read_pos_scores[traj.read_name][pos]

                # Find this read's scores at neighboring positions
                max_neighbor = my_score
                for offset in range(-half_k, half_k + 1):
                    if offset == 0:
                        continue
                    nbr_pos = pos + offset
                    nbr_score = read_pos_scores.get(traj.read_name, {}).get(nbr_pos)
                    if nbr_score is not None and nbr_score > max_neighbor:
                        max_neighbor = nbr_score

                # Attenuation: if not the local max, scale down
                if max_neighbor > my_score and max_neighbor > 0.1:
                    ratio = my_score / max_neighbor
                    # Quadratic suppression: if ratio=0.9, keep 81%; if ratio=0.5, keep 25%
                    adjusted_native.append(my_score * ratio)
                else:
                    adjusted_native.append(my_score)

            if adjusted_native:
                results[(contig, pos)] = float(np.mean(adjusted_native))
            else:
                results[(contig, pos)] = 0.0

    return results


print('Computing read-level consistency scores...')
read_consist_scores = read_consistency_deconv(hier_results, contig_results, kmer_k=KMER_K)

df['mod_ratio_read_consist'] = df.apply(
    lambda r: read_consist_scores.get((r['contig'], r['position']), 0.0),
    axis=1,
)
print(f'Read-consistency: mean attenuation = '
      f'{1 - df["mod_ratio_read_consist"].sum() / df["mean_p_mod_orig"].clip(lower=1e-10).sum():.3f}')

---
## 9. Visual Comparison: Signal Tracks Around Known Sites

Plot the modification profile around known modification sites, comparing original vs. each deconvolution method.
This is the most informative visualization — it directly shows whether bleed is reduced.

In [ ]:
# Select the best-performing LASSO alpha for comparison (or default)
best_lasso_col = 'mod_ratio_lasso_0.01'  # will update after ROC analysis

METHODS_TO_PLOT = [
    ('mod_ratio_orig',          'Original',          '#78909C'),
    ('mod_ratio_peak_soft',     'Peak-calling (soft)', '#42A5F5'),
    (best_lasso_col,            'LASSO (α=0.01)',    '#66BB6A'),
    ('mod_ratio_read_consist',  'Read consistency',   '#FFA726'),
]

# Plot around each known modification site (±10 positions)
WINDOW = 10

known_sites = df[df['y_true'] == 1].sort_values(['contig', 'position'])
n_sites = len(known_sites)

if n_sites == 0:
    print('No known modification sites found in the data.')
else:
    ncols = min(3, n_sites)
    nrows = (n_sites + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows),
                             squeeze=False)

    for idx, (_, site_row) in enumerate(known_sites.iterrows()):
        ax = axes[idx // ncols][idx % ncols]
        contig = site_row['contig']
        site_pos = site_row['position']
        mod_type = site_row['mod_type']

        sub = df[(df['contig'] == contig) &
                 (df['position'] >= site_pos - WINDOW) &
                 (df['position'] <= site_pos + WINDOW)].sort_values('position')

        for col, label, color in METHODS_TO_PLOT:
            if col in sub.columns:
                ax.plot(sub['position'], sub[col], 'o-', label=label,
                        color=color, markersize=3, linewidth=1.2)

        ax.axvline(site_pos, color='red', linestyle='--', alpha=0.7, linewidth=1)
        ax.set_title(f'{contig}:{site_pos} ({mod_type})', fontsize=9)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel('Position')
        ax.set_ylabel('Mod ratio')

        # Shade the k-mer bleed zone
        ax.axvspan(site_pos - HALF_K, site_pos + HALF_K,
                   alpha=0.08, color='red', label='k-mer zone')

    # Add legend to first plot
    axes[0][0].legend(fontsize=6, loc='upper right')

    # Hide empty axes
    for idx in range(n_sites, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle('Modification Profiles Around Known Sites', fontsize=12, y=1.01)
    fig.tight_layout()
    plt.show()

---
## 10. ROC and PR Curves: All Methods

In [ ]:
y_true = df['y_true'].values

ALL_METHODS = [
    ('mod_ratio_orig',          'Original',             '#78909C'),
    ('mod_ratio_peak_hard',     'Peak-calling (hard)',   '#1E88E5'),
    ('mod_ratio_peak_soft',     'Peak-calling (soft)',   '#42A5F5'),
    ('mod_ratio_read_consist',  'Read consistency',      '#FFA726'),
]

# Add LASSO columns
for alpha in ALPHA_VALUES:
    col = f'mod_ratio_lasso_{alpha}'
    ALL_METHODS.append((col, f'LASSO α={alpha}', None))

# Auto-assign colors for LASSO
lasso_colors = plt.cm.Greens(np.linspace(0.3, 0.9, len(ALPHA_VALUES)))
for i, alpha in enumerate(ALPHA_VALUES):
    col = f'mod_ratio_lasso_{alpha}'
    for j, (c, l, clr) in enumerate(ALL_METHODS):
        if c == col:
            ALL_METHODS[j] = (c, l, lasso_colors[i])

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))

roc_summary = []
for col, label, color in ALL_METHODS:
    if col not in df.columns:
        continue
    scores = df[col].values
    valid = ~np.isnan(scores)
    if len(np.unique(y_true[valid])) < 2:
        continue

    fpr, tpr, _ = roc_curve(y_true[valid], scores[valid])
    roc_a = auc(fpr, tpr)
    pr_prec, pr_rec, _ = precision_recall_curve(y_true[valid], scores[valid])
    pr_a = average_precision_score(y_true[valid], scores[valid])

    ax_roc.plot(fpr, tpr, label=f'{label} (AUC={roc_a:.3f})',
                color=color, linewidth=1.5)
    ax_pr.plot(pr_rec, pr_prec, label=f'{label} (AP={pr_a:.3f})',
               color=color, linewidth=1.5)

    roc_summary.append({'method': label, 'roc_auc': roc_a, 'avg_precision': pr_a,
                        'column': col})

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax_roc.set(xlabel='FPR', ylabel='TPR', title='ROC Curves')
ax_roc.legend(fontsize=7)

baseline_pr = y_true.sum() / len(y_true)
ax_pr.axhline(baseline_pr, color='k', linestyle='--', alpha=0.3)
ax_pr.set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves')
ax_pr.legend(fontsize=7)

fig.tight_layout()
plt.show()

pd.DataFrame(roc_summary).sort_values('roc_auc', ascending=False)

---
## 11. Key Metric: False Positive Rate in K-mer Flanks

This is the **most directly relevant** metric for the bleed problem.
For each known modification site, count how many flanking positions
(within ±2bp) are called as modified at various thresholds.

In [ ]:
def flank_fp_analysis(df, method_col, thresholds=[0.1, 0.2, 0.3, 0.5]):
    """Count false positive calls in k-mer flanking positions."""
    known_positions = set()
    flank_positions = set()

    for _, row in df[df['y_true'] == 1].iterrows():
        contig = row['contig']
        pos = row['position']
        known_positions.add((contig, pos))
        for offset in range(-HALF_K, HALF_K + 1):
            if offset != 0:
                flank_positions.add((contig, pos + offset))

    # Remove known sites from flanks (in case two mods are adjacent)
    flank_positions -= known_positions

    # Get flank rows
    flank_mask = df.apply(
        lambda r: (r['contig'], r['position']) in flank_positions, axis=1
    )
    flank_df = df[flank_mask]

    records = []
    for thresh in thresholds:
        n_fp = (flank_df[method_col] >= thresh).sum()
        n_total = len(flank_df)
        records.append({
            'threshold': thresh,
            'flank_FP': n_fp,
            'flank_total': n_total,
            'flank_FPR': n_fp / max(n_total, 1),
        })
    return pd.DataFrame(records)


METHOD_COLS_FOR_FLANK = [
    ('mod_ratio_orig',          'Original'),
    ('mod_ratio_peak_hard',     'Peak-calling (hard)'),
    ('mod_ratio_peak_soft',     'Peak-calling (soft)'),
    ('mod_ratio_lasso_0.01',    'LASSO α=0.01'),
    ('mod_ratio_read_consist',  'Read consistency'),
]

flank_results = []
for col, label in METHOD_COLS_FOR_FLANK:
    if col not in df.columns:
        continue
    fpa = flank_fp_analysis(df, col)
    fpa['method'] = label
    flank_results.append(fpa)

flank_df_all = pd.concat(flank_results, ignore_index=True)

# Pivot for display
pivot = flank_df_all.pivot(index='threshold', columns='method', values='flank_FP')
print('Flank FP counts at each threshold:')
print(pivot.to_string())
print()

# Grouped bar chart
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax)
ax.set_xlabel('Mod ratio threshold')
ax.set_ylabel('Flank false positives')
ax.set_title('False Positives in K-mer Flanking Positions')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

---
## 12. Sensitivity Check: Are True Sites Preserved?

Critical sanity check — make sure deconvolution doesn't kill true modification signals.

In [ ]:
known_df = df[df['y_true'] == 1].copy()

display_cols = ['contig', 'position', 'kmer', 'mod_type',
                'mod_ratio_orig', 'mod_ratio_peak_soft',
                'mod_ratio_lasso_0.01', 'mod_ratio_read_consist']
display_cols = [c for c in display_cols if c in known_df.columns]

print('Scores at known modification sites:')
print(known_df[display_cols].to_string(index=False))

# Check for sites where deconvolution dropped the score significantly
for col, label in [('mod_ratio_peak_soft', 'Peak-soft'),
                    ('mod_ratio_lasso_0.01', 'LASSO'),
                    ('mod_ratio_read_consist', 'Read-consist')]:
    if col not in known_df.columns:
        continue
    drops = known_df[known_df[col] < known_df['mod_ratio_orig'] * 0.5]
    if len(drops) > 0:
        print(f'\n⚠ {label}: {len(drops)} known sites lost >50% of their score:')
        print(drops[['contig', 'position', 'mod_type', 'mod_ratio_orig', col]].to_string(index=False))
    else:
        print(f'\n✓ {label}: all known sites preserved (no >50% drops)')

---
## 13. LASSO Alpha Selection via BIC

Automatically select the best LASSO alpha using BIC (no labeled data needed).

In [ ]:
def lasso_bic(positions, mod_ratios, kmer_k=5, alphas=None):
    """Select LASSO alpha by BIC."""
    if alphas is None:
        alphas = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]
    half_k = kmer_k // 2
    n_pos = len(positions)

    all_bases = set()
    for p in positions:
        for offset in range(-half_k, half_k + 1):
            all_bases.add(p + offset)
    all_bases = sorted(all_bases)
    base_to_col = {b: j for j, b in enumerate(all_bases)}
    n_bases = len(all_bases)

    X = np.zeros((n_pos, n_bases), dtype=np.float64)
    for i, p in enumerate(positions):
        for offset in range(-half_k, half_k + 1):
            base_pos = p + offset
            if base_pos in base_to_col:
                X[i, base_to_col[base_pos]] = 1.0

    y = mod_ratios.copy()
    results = []

    for alpha in alphas:
        lasso = Lasso(alpha=alpha, positive=True, max_iter=10000)
        lasso.fit(X, y)
        y_pred = lasso.predict(X)
        rss = np.sum((y - y_pred) ** 2)
        k_params = np.sum(lasso.coef_ > 0)  # number of non-zero coefficients
        n = len(y)
        # BIC = n*log(RSS/n) + k*log(n)
        bic = n * np.log(rss / n + 1e-300) + k_params * np.log(n)
        results.append({'alpha': alpha, 'bic': bic, 'n_nonzero': int(k_params),
                        'rss': rss})

    return pd.DataFrame(results)


# Run BIC selection per contig
bic_frames = []
for contig in df['contig'].unique():
    mask = df['contig'] == contig
    sub = df[mask].sort_values('position')
    bic_df = lasso_bic(sub['position'].values, sub['mod_ratio_orig'].values)
    bic_df['contig'] = contig
    bic_frames.append(bic_df)

bic_all = pd.concat(bic_frames, ignore_index=True)

# Aggregate BIC across contigs
bic_agg = bic_all.groupby('alpha').agg({'bic': 'sum', 'n_nonzero': 'sum'}).reset_index()
best_alpha = bic_agg.loc[bic_agg['bic'].idxmin(), 'alpha']
print(f'BIC-selected alpha: {best_alpha}')
print()
print(bic_agg.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(bic_agg['alpha'], bic_agg['bic'], 'o-')
ax.axvline(best_alpha, color='red', linestyle='--', label=f'Best α={best_alpha}')
ax.set_xscale('log')
ax.set_xlabel('LASSO α')
ax.set_ylabel('BIC (summed across contigs)')
ax.set_title('LASSO Alpha Selection')
ax.legend()
fig.tight_layout()
plt.show()

---
## 14. Zoomed Comparison: Your Reported Region (ecoli16S:511–529)

Direct comparison to the data you showed, with known sites at 516 (Ψ) and 527 (m7G).

In [ ]:
ZOOM_CONTIG = 'ecoli16S'
ZOOM_START, ZOOM_END = 511, 529

zoom = df[(df['contig'] == ZOOM_CONTIG) &
          (df['position'] >= ZOOM_START) &
          (df['position'] <= ZOOM_END)].sort_values('position').copy()

if len(zoom) == 0:
    print(f'No data for {ZOOM_CONTIG}:{ZOOM_START}-{ZOOM_END}')
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)

    plot_methods = [
        (axes[0, 0], 'mod_ratio_orig',         'Original (no correction)', '#78909C'),
        (axes[0, 1], 'mod_ratio_peak_soft',     'Peak-calling (soft)',      '#42A5F5'),
        (axes[1, 0], f'mod_ratio_lasso_{best_alpha}'
                     if f'mod_ratio_lasso_{best_alpha}' in df.columns
                     else 'mod_ratio_lasso_0.01',
                     f'LASSO (α={best_alpha})',  '#66BB6A'),
        (axes[1, 1], 'mod_ratio_read_consist',  'Read consistency',         '#FFA726'),
    ]

    known_pos_in_range = [516, 527]  # known mod sites

    for ax, col, title, color in plot_methods:
        if col not in zoom.columns:
            ax.set_title(f'{title} (N/A)')
            continue

        bars = ax.bar(zoom['position'], zoom[col], color=color, alpha=0.7, width=0.8)

        # Highlight known sites
        for p in known_pos_in_range:
            ax.axvline(p, color='red', linestyle='--', alpha=0.6, linewidth=1)
            # Shade k-mer zone
            ax.axvspan(p - HALF_K, p + HALF_K, alpha=0.06, color='red')

        ax.set_title(title, fontsize=11)
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel('Mod ratio')

    axes[1, 0].set_xlabel('Position')
    axes[1, 1].set_xlabel('Position')

    fig.suptitle(f'{ZOOM_CONTIG}:{ZOOM_START}–{ZOOM_END}  '
                 f'(Known mods: 516=Ψ, 527=m7G)', fontsize=13)
    fig.tight_layout()
    plt.show()

    # Print table
    show_cols = ['position', 'kmer', 'mod_ratio_orig', 'mod_ratio_peak_soft',
                 'mod_ratio_read_consist', 'y_true']
    lasso_col = f'mod_ratio_lasso_{best_alpha}' if f'mod_ratio_lasso_{best_alpha}' in zoom.columns else 'mod_ratio_lasso_0.01'
    show_cols.insert(4, lasso_col)
    show_cols = [c for c in show_cols if c in zoom.columns]
    print(zoom[show_cols].to_string(index=False))

---
## 15. Summary Table

In [ ]:
# Compile final summary
if len(roc_summary) > 0:
    summary_df = pd.DataFrame(roc_summary)

    # Add flank FP count at threshold=0.3
    flank_at_03 = flank_df_all[flank_df_all['threshold'] == 0.3][['method', 'flank_FP']]
    summary_df = summary_df.merge(flank_at_03, on='method', how='left')

    # Add mean score at known sites
    for _, row in summary_df.iterrows():
        col = row['column']
        if col in known_df.columns:
            summary_df.loc[summary_df['column'] == col, 'mean_at_known'] = \
                known_df[col].mean()

    print(summary_df[['method', 'roc_auc', 'avg_precision', 'flank_FP',
                       'mean_at_known']].sort_values('roc_auc', ascending=False).to_string(index=False))
else:
    print('No ROC summary available — run Section 10 first.')

---
## 16. Conclusions and Next Steps

Compare the methods on:

1. **ROC AUC / AP** — overall discrimination (should stay same or improve)
2. **Flank FP count** — the target metric (should decrease dramatically)
3. **Score at known sites** — sensitivity preservation (should stay high)

The ideal method reduces flank FPs while preserving true site scores.

### Things to watch for:
- Adjacent modifications (e.g., ecoli23S 2604/2605) — do methods incorrectly suppress one?
- Low-signal modifications — are they disproportionately affected?
- LASSO alpha sensitivity — does BIC-selected alpha generalize?